#**Tugas 6 : Sistem pencarian dokumen dengan penerapan Singular Value Decomposition (SVD)**

- membuat sistem pencarian dokumen
- reduksi dimensi SVD
- data baru juga di reduksi
- mencari kemiripan ecludian distance

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

data_path = ("/content/drive/My Drive/PPW-A/report/Tugas-PPW/hasil_preprocesing.csv")
news_data = pd.read_csv(data_path)

news_data

,judul,isi,tanggal,kategori,Hasil cleansing,Hasil case_folding,tokenize,Hasil stopword
0,18 Tim yang Lolos FFWS Global Finals 2024 di B...,Jakarta - Rangkaian pertandingan di FFWS SEA 2...,"Rabu, 16 Okt 2024 19:30 WIB",Games,Jakarta Rangkaian pertandingan di FFWS SEA F...,jakarta rangkaian pertandingan di ffws sea f...,"['jakarta', 'rangkaian', 'pertandingan', 'di',...",jakarta rangkaian pertandingan ffws sea fall s...
1,Ampun Bang Jago! Zuckerberg Klaim Pemain Handa...,"Jakarta - CEO Meta Mark Zuckerberg, cukup perc...","Rabu, 16 Okt 2024 17:50 WIB",Games,Jakarta CEO Meta Mark Zuckerberg cukup percay...,jakarta ceo meta mark zuckerberg cukup percay...,"['jakarta', 'ceo', 'meta', 'mark', 'zuckerberg...",jakarta ceo meta mark zuckerberg percaya kemam...
2,"20 Game Horor Terbaik Tahun 2000an, Udah Perna...",Jakarta - Game horor selalu punya daya tarikny...,"Rabu, 16 Okt 2024 14:20 WIB",Games,Jakarta Game horor selalu punya daya tariknya...,jakarta game horor selalu punya daya tariknya...,"['jakarta', 'game', 'horor', 'selalu', 'punya'...",jakarta game horor daya tariknya penikmatnya b...
3,"Begini Cara Dapat Skin Zhou Yu HOK Gratis, Gam...",Jakarta - Ternyata cara dapat skin Zhou Yu Hon...,"Rabu, 16 Okt 2024 12:15 WIB",Games,Jakarta Ternyata cara dapat skin Zhou Yu Hono...,jakarta ternyata cara dapat skin zhou yu hono...,"['jakarta', 'ternyata', 'cara', 'dapat', 'skin...",jakarta skin zhou yu honor of kings hok gratis...
4,"Gagal Juara FFWS SEA 2024 Fall, Wakil RI: Kita...","Jakarta - Tampil di depan ribuan pendukungnya,...","Selasa, 15 Okt 2024 17:36 WIB",Games,Jakarta Tampil di depan ribuan pendukungnya t...,jakarta tampil di depan ribuan pendukungnya t...,"['jakarta', 'tampil', 'di', 'depan', 'ribuan',...",jakarta tampil ribuan pendukungnya wakil indon...
...,...,...,...,...,...,...,...,...
95,Rating Pemain China Vs Indonesia: Thom Haye Te...,Jakarta - China vs Indonesia tuntas 2-1 di lan...,"Rabu, 16 Okt 2024 12:00 WIB",Sepak Bola,Jakarta China vs Indonesia tuntas di lanjuta...,jakarta china vs indonesia tuntas di lanjuta...,"['jakarta', 'china', 'vs', 'indonesia', 'tunta...",jakarta china vs indonesia tuntas lanjutan kua...
96,China Vs Indonesia: Coret Eliano Reijnders Blu...,Qingdao - Keputusan mencoret Eliano Reijnders ...,"Rabu, 16 Okt 2024 11:40 WIB",Sepak Bola,Qingdao Keputusan mencoret Eliano Reijnders s...,qingdao keputusan mencoret eliano reijnders s...,"['qingdao', 'keputusan', 'mencoret', 'eliano',...",qingdao keputusan mencoret eliano reijnders me...
97,Dumfries Akui Performanya Lebih Baik di Timnas...,"Jakarta - Denzel Dumfries mengakui, performany...","Rabu, 16 Okt 2024 11:20 WIB",Sepak Bola,Jakarta Denzel Dumfries mengakui performanya ...,jakarta denzel dumfries mengakui performanya ...,"['jakarta', 'denzel', 'dumfries', 'mengakui', ...",jakarta denzel dumfries mengakui performanya t...
98,Hasil Lengkap-Klasemen Kualifikasi Piala Dunia...,Jakarta - Matchday 4 di babak ketiga kualifika...,"Rabu, 16 Okt 2024 11:00 WIB",Sepak Bola,Jakarta Matchday di babak ketiga kualifikasi...,jakarta matchday di babak ketiga kualifikasi...,"['jakarta', 'matchday', 'di', 'babak', 'ketiga...",jakarta matchday babak ketiga kualifikasi pial...


In [ ]:
# Ambil kolom dokumen berita
documents = news_data['Hasil stopword'].tolist()

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Mendapatkan fitur (kata-kata) dari TF-IDF
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Mengonversi matriks TF-IDF menjadi DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)

# Tampilkan hasil TF-IDF
print("\nHasil TF-IDF dalam bentuk DataFrame:")
tfidf_df


Hasil TF-IDF dalam bentuk DataFrame:


,aaa,abaaax,abaay,abdulla,abduweli,abduwelli,about,absen,abu,ac,...,zero,zhang,zhou,zombie,zona,zone,zoo,zubimendi,zuckerberg,zwolle
0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.555878,0.000000
2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.025206,0.000000,0.0,0.025206,0.000000,0.000000,0.000000
3,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.463359,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,0.0,0.0,0.089759,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
96,0.0,0.0,0.0,0.0,0.000000,0.048644,0.0,0.000000,0.0,0.0,...,0.0,0.036141,0.000000,0.000000,0.034519,0.0,0.000000,0.000000,0.000000,0.059771
97,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
98,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.293640,0.0,0.000000,0.000000,0.000000,0.000000


**Singular Value Decomposition (SVD)**

In [ ]:
# Menggunakan SVD untuk reduksi dimensi
n_components = 99
svd = TruncatedSVD(n_components=n_components)
svd_matrix = svd.fit_transform(tfidf_matrix)

# Mengubah hasil SVD menjadi DataFrame
svd_df = pd.DataFrame(svd_matrix, columns=[f'feature {i+1}' for i in range(n_components)])
svd_df['kategori'] = news_data['kategori'].values
# Menampilkan hasil
print("Hasil reduksi dimensi dengan SVD:")
svd_df

Hasil reduksi dimensi dengan SVD:


,feature 1,feature 2,feature 3,feature 4,feature 5,feature 6,feature 7,feature 8,feature 9,feature 10,...,feature 91,feature 92,feature 93,feature 94,feature 95,feature 96,feature 97,feature 98,feature 99,kategori
0,0.485972,-0.397006,-0.091042,-0.209983,-0.080212,-0.152371,0.049150,0.013784,-0.001869,-0.005952,...,0.004836,-0.024326,0.019259,-0.000402,0.005825,0.003606,0.004569,0.025975,0.011168,Games
1,0.061095,-0.013025,0.003460,-0.020003,0.058970,0.116627,-0.158594,-0.043621,0.012218,-0.067623,...,0.006421,-0.003008,-0.000856,-0.001069,-0.007327,0.000154,0.000882,-0.002550,0.000806,Games
2,0.043955,-0.027072,0.000610,-0.067151,0.178105,0.437597,0.316979,0.115034,0.057363,-0.171429,...,0.004428,-0.005097,0.000266,-0.000095,-0.000478,-0.000161,-0.002063,0.008183,0.003298,Games
3,0.081272,-0.056543,-0.007903,0.006373,0.041068,0.081331,-0.077024,-0.026504,0.006277,0.060420,...,0.000543,-0.001065,0.001771,0.000606,-0.000627,0.003487,0.001457,0.003118,0.000668,Games
4,0.283298,-0.236096,-0.042034,-0.196717,-0.018579,-0.044174,-0.025302,-0.010250,-0.003405,-0.046209,...,0.002988,-0.003311,-0.003206,-0.001560,-0.000456,-0.000708,0.005530,-0.003203,-0.001022,Games
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.417327,0.386491,-0.023466,0.004512,-0.063981,0.000640,0.045112,-0.086001,0.032507,-0.001527,...,-0.000975,0.001412,0.006111,-0.007053,-0.010376,-0.002519,0.000728,-0.003976,-0.001316,Sepak Bola
96,0.351679,0.364925,-0.015660,-0.003685,-0.055741,-0.003596,0.055699,-0.080195,-0.056829,-0.074680,...,-0.000949,0.005953,0.007469,-0.001269,-0.002181,-0.003216,0.000853,-0.003467,-0.002667,Sepak Bola
97,0.069738,0.042609,0.004084,0.010686,0.031999,-0.011496,-0.017270,0.003994,0.090390,-0.009614,...,-0.001061,0.003431,-0.002600,-0.002911,0.000536,0.001331,-0.001137,-0.004093,-0.002998,Sepak Bola
98,0.291987,0.192205,-0.017532,0.045260,-0.022631,-0.000224,-0.047140,0.108819,0.035865,0.155576,...,-0.008081,0.019445,-0.005280,0.009369,-0.000513,-0.003614,-0.000194,-0.007614,-0.001990,Sepak Bola


**Implementasi**

In [ ]:

def search_similar_documents(query_text, top_k=5):

    # Transformasi query text
    query_tfidf = tfidf_vectorizer.transform([query_text])
    query_svd = svd.transform(query_tfidf)

    # Hitung jarak Euclidean antara query dan dokumen
    distances = euclidean_distances(query_svd, svd_matrix)

    # Ambil indeks dokumen terdekat
    closest_indices = np.argsort(distances[0])[:top_k]
    similarities = distances[0][closest_indices]

    return closest_indices, similarities

In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pastikan untuk mendownload stopwords jika belum
import nltk
nltk.download('stopwords')
nltk.download('punkt')

# Fungsi untuk melakukan preprocessing
def preprocess_text(text):
    # Cleaning: Menghapus angka dan tanda baca
    text = re.sub(r'\d+', '', text)  # Hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # Hapus tanda baca

    # Casefolding: Mengubah ke huruf kecil
    text = text.lower()

    # Tokenizing
    words = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words("indonesian"))  # Ganti dengan 'english' jika teks berbahasa Inggris
    words = [word for word in words if word not in stop_words]

    # Gabungkan kembali menjadi string
    return ' '.join(words)

# Input teks dan preprocessing
query_text = input("Masukkan teks: ")
processed_query_text = preprocess_text(query_text)

# Lakukan pencarian dengan teks yang telah dipreproses
top_k = 5  # Jumlah dokumen yang diinginkan
indices, distances = search_similar_documents(processed_query_text, top_k)

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    print(f"Dokumen ke-{idx + 1}: {documents[idx]}")
    print(f"Jarak Euclidean: {distance:.4f}\n")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Masukkan teks: menang 

Hasil pencarian untuk query: menang
Dokumen ke-58: jakarta hasil imbang melawan australia menyadarkan jepang laga grup c kualifikasi piala dunia sepenuhnya mudah jepang menang jepang bermain imbang australia laga kualifikasi piala dunia zona asia ronde ketiga grup saitama stadium selasa malam wib tuan rumah tertinggal akibat gol bunuh shogo taniguchi jepang selamat kekalahan gol bunuh cameron burgess membobol gawang hasil imbang menghentikan rentetan kemenangan jepang grup c samurai biru menang laga laga jepang tampil sempurna bikin gol kebobolan gol australia gol bersarang gawang jepang grup c jepang puncak grup c poin australia peringkat poin hasil imbang melawan australia menyadarkan pelatih jepang hajime moriyasu grup c sepenuhnya mudah bertekad tim laga laga jepang bersua indonesia indonesia vs jepang stadion utama gelora karno belajar sulit menang babak kualifikasi moriyasu dikutip supersport bermain melawan australia menyadari mudah menang babak kualifika